In [ ]:
import pandas as pd
from sqlalchemy import create_engine
import datetime as dt
import time as tm
import numpy as np
from pymongo import MongoClient
from math import radians, cos, sin,asin, sqrt
import pymysql
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 100000)
import json
import urllib.request
import requests
from datetime import datetime,timedelta

import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('seaborn')

In [ ]:
import os
from config import *  # all DB / API handles come from environment variables, see config.py


db = client.get_database('telematics_db')






In [ ]:
gettime=lambda x: pd.Timestamp(x).timestamp()
getdate=lambda x: pd.datetime.fromtimestamp(x).date()
getDay=lambda x: pd.datetime.fromtimestamp(x).date()
dttoday=int(tm.time()-tm.time()%86400-19800)



In [ ]:
def action_analysis(vid): 
    if 1>0:
    #try:
        
        device_health=pd.read_sql("select * from device_health where vehicle_id={}  order by day desc limit 9 ".format(vid),con=engine).drop_duplicates(['vehicle_id','day'])
        #device_health=device_health[device_health['total_raw_pings']>5][device_health['heart_beat_pings']>5]
        
        wire_issues=pd.read_sql("select * from wire_issues where vehicle_id={}  order by day desc limit 9".format(vid),con=engine).drop_duplicates(['vehicle_id','day'])
        
        device_health['day'] = pd.to_datetime(device_health['day'], utc = True)
        wire_issues['day'] = pd.to_datetime(wire_issues['day'], utc = True)

        complete_health=pd.merge(wire_issues,device_health,how='inner',on='day')
        print(vid,str(len(complete_health)))
        threshold=4

        action_values=[]

        if len(complete_health)>threshold:
            print(vid)
            intentional_wire_cut_count=len(complete_health[complete_health['intentional_wire_cut']==True])
            wire_loose_count=len(complete_health[complete_health['wire_loose']==True])
            charge_to_ign_count=len(complete_health[complete_health['charge_to_ign']==True])
            ign_wire_cut_count=len(complete_health[complete_health['ign_wire_cut']==True])
            ign_to_charge_count=len(complete_health[complete_health['ign_to_charge']==True])
            maintainence_wire_cut_count=len(complete_health[complete_health['maintainence_wire_cut']==True])
            jump_percent_avg=complete_health['jump_percent'].mean()
            total_distace_travelled_avg=complete_health['total_distace_travelled'].mean()


            if (wire_loose_count<2)&(intentional_wire_cut_count<2)&(charge_to_ign_count<5)&(jump_percent_avg>5)&(ign_wire_cut_count<2):
                action_values.append([vid,datetime.now().date(),'replace_device',intentional_wire_cut_count,wire_loose_count,charge_to_ign_count,ign_wire_cut_count,ign_to_charge_count,maintainence_wire_cut_count])

            elif (wire_loose_count>2)|(intentional_wire_cut_count>2)|(charge_to_ign_count>2)|(maintainence_wire_cut_count>2)|(ign_wire_cut_count>2):
                action_values.append([vid,datetime.now().date(),'escalation_required',intentional_wire_cut_count,wire_loose_count,charge_to_ign_count,ign_wire_cut_count,ign_to_charge_count,maintainence_wire_cut_count])
            else:
                
                action_values.append([vid,datetime.now().date(),'device_working_fine',intentional_wire_cut_count,wire_loose_count,charge_to_ign_count,ign_wire_cut_count,ign_to_charge_count,maintainence_wire_cut_count])
            
            action=pd.DataFrame(action_values,columns=['vehicle_id','date_of_issue_identified','action_required','intentional_wire_cut_count','wire_loose_count','charge_to_ign_count','ign_wire_cut_count','ign_to_charge_count','maintainence_wire_cut_count'])

        else:
            action=pd.DataFrame()

        return action
    else:
    #except:
        return None


In [ ]:
vehicle_details=pd.read_sql("select vehicle_id,vehicle_number,model_name,operator_name,operator_phone_number from vehicle_details where operator_phone_number='XXXXXXXXXX'",con=engine)

In [ ]:
from multiprocessing.dummy import Pool as ThreadPool
pool = ThreadPool(10)
results2=pool.map(action_analysis,list(vehicle_details['vehicle_id']))
pool.close()
pool.join()


In [ ]:
muhaha=pd.concat(results2)

In [ ]:
len(muhaha)

In [ ]:
no_data_paf_vehicles=set(vehicle_details['vehicle_id'])-set(muhaha['vehicle_id'])

In [ ]:
muhaha=pd.merge(muhaha,vehicle_details,how='left',on='vehicle_id')

In [ ]:
no_data=vehicle_details[vehicle_details['vehicle_id'].isin(no_data_paf_vehicles)]

In [ ]:
no_data.to_excel("No_data_CUSTOMER.xlsx")
muhaha.to_excel("CUSTOMER_proactive_escalation_data_5pct.xlsx")

In [ ]:
a=pd.read_sql("select * from device_health where vehicle_id=100002  order by day desc limit 9 ",con=engine)
a.drop_duplicates(['vehicle_id','day'],inplace=True)

In [ ]:
b=pd.read_sql("select * from wire_issues where vehicle_id=100002  order by day desc limit 9 ",con=engine)
b.drop_duplicates(['vehicle_id','day'],inplace=True)

In [ ]:
a['day']=pd.to_datetime(a['day'], utc = True)
b['day']=pd.to_datetime(b['day'], utc = True)



In [ ]:
c=pd.merge(a,b,how='inner',on='day')

In [ ]:
c

In [ ]:
action_analysis(100002)